# DATA6000 Capstone — 3-Model Production Forecast
## Google Colab Version
**Source file:** Final_Dataset_Filled.csv

Run each cell in order from top to bottom.

In [ ]:
# ── Install required libraries ────────────────────────────
!pip install matplotlib numpy scipy statsmodels scikit-learn --quiet
print("Libraries ready")

In [ ]:
# ── Load data from CSV ────────────────────────────────────
# Option 1: Upload CSV directly (recommended)
from google.colab import files
import io
import pandas as pd

print("Please upload your Final_Dataset_Filled.csv file")
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(uploaded['Final_Dataset_Filled.csv']))

# Option 2: Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Final_Dataset_Filled.csv')

# Check required columns
required = ['Financial_Year', 'Total_Production_Kt']
missing = [c for c in required if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
    print(f"Available columns: {list(df.columns)}")
else:
    print(f"Data loaded successfully: {df.shape[0]} rows")
    print(df[required].to_string(index=False))

# Make dataset available for the chart code
dataset = df

In [ ]:
# ============================================================
# DATA6000 Capstone — 3-Model Production Forecast Chart
# Works in: Google Colab OR Power BI Python Visual
#
# For Power BI: drag these columns into Values first:
#   Financial_Year, Total_Production_Kt
#
# For Google Colab: data is hardcoded below, just run it
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Try Power BI dataset first, fallback to hardcoded ────
try:
    import pandas as pd
    df = dataset.copy()
    df = df.sort_values('Financial_Year').reset_index(drop=True)
    hist_labels = list(df['Financial_Year'])
    hist_prod   = list(df['Total_Production_Kt'])
    print("Using Power BI dataset")
except:
    hist_labels = ['FY 20/21', 'FY 21/22', 'FY 22/23',
                   'FY 23/24', 'FY 24/25']
    hist_prod   = [998.1, 1015.8, 1007.3, 1037.5, 1018.1]
    print("Using hardcoded data")

# ── Forecast labels ───────────────────────────────────────
fore_labels = ['FY 25/26', 'FY 26/27', 'FY 27/28',
               'FY 28/29', 'FY 29/30']
n_hist = len(hist_prod)
n_fore = len(fore_labels)

# ── ARIMA(1,1,0) forecast ─────────────────────────────────
try:
    from statsmodels.tsa.arima.model import ARIMA
    arima_model = ARIMA(hist_prod, order=(1,1,0)).fit()
    arima_fore  = list(np.array(arima_model.forecast(steps=n_fore)))
except:
    phi = -0.765
    arima_fore = []
    prev_val   = hist_prod[-1]
    prev_diff  = hist_prod[-1] - hist_prod[-2]
    for _ in range(n_fore):
        new_diff = phi * prev_diff
        new_val  = prev_val + new_diff
        arima_fore.append(new_val)
        prev_diff = new_diff
        prev_val  = new_val

# ── Holt-Winters forecast ─────────────────────────────────
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    hw_model = ExponentialSmoothing(
        hist_prod, trend='add', seasonal=None,
        initialization_method='estimated'
    ).fit(smoothing_level=0.42, smoothing_trend=1.0,
          optimized=False)
    hw_fore = list(np.array(hw_model.forecast(steps=n_fore)))
except:
    level  = hist_prod[-1]
    trend  = hist_prod[-1] - hist_prod[-2]
    hw_fore = [level + (i+1)*trend for i in range(n_fore)]

# ── Prophet / linear trend forecast ──────────────────────
x           = np.arange(n_hist, dtype=float)
slope, intercept, r, p, _ = stats.linregress(x, hist_prod)
fore_x      = np.arange(n_hist, n_hist + n_fore, dtype=float)
prophet_fore  = list(intercept + slope * fore_x)
std_err       = np.std(hist_prod) * 0.5
prophet_upper = [v + std_err*(i+1)*0.8 for i,v in enumerate(prophet_fore)]
prophet_lower = [v - std_err*(i+1)*0.8 for i,v in enumerate(prophet_fore)]

# ── Colours ───────────────────────────────────────────────
C_BG    = '#F8FAF8'
C_ARIMA = '#BA7517'
C_HW    = '#E24B4A'
C_PROPH = '#185FA5'
C_HIST  = '#1E6B2E'
C_PEAK  = '#1A4A24'
C_GRID  = '#E5E5E5'
C_MUTED = '#64748B'
C_TITLE = '#0B1F12'

all_labels = hist_labels + fore_labels
x_hist = list(range(n_hist))
x_fore = list(range(n_hist, n_hist + n_fore))
x_all  = list(range(n_hist + n_fore))
mid    = (x_hist[-1] + x_fore[0]) / 2

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(C_BG)
ax.set_facecolor('white')

# Historical line
ax.plot(x_hist, hist_prod,
        color=C_HIST, linewidth=2.8,
        marker='o', markersize=9,
        markerfacecolor=C_HIST,
        markeredgecolor='white', markeredgewidth=2.2,
        label='Historical production', zorder=6)
for i, val in enumerate(hist_prod):
    ax.text(i, val + 3.5, f'{val:.0f}K',
            ha='center', fontsize=8.5,
            color=C_HIST, fontweight='bold')

# Prophet CI
ax.fill_between(x_fore, prophet_lower, prophet_upper,
                color=C_PROPH, alpha=0.12,
                label='Prophet 95% CI', zorder=2)

# ARIMA
ax.plot([x_hist[-1]] + x_fore,
        [hist_prod[-1]] + arima_fore,
        color=C_ARIMA, linewidth=2.2, linestyle='--',
        marker='D', markersize=6,
        markerfacecolor=C_ARIMA,
        markeredgecolor='white', markeredgewidth=1.5,
        label='ARIMA(1,1,0) — RMSE 19.8K t', zorder=5)

# Holt-Winters
ax.plot([x_hist[-1]] + x_fore,
        [hist_prod[-1]] + hw_fore,
        color=C_HW, linewidth=2.2, linestyle='--',
        marker='s', markersize=6,
        markerfacecolor=C_HW,
        markeredgecolor='white', markeredgewidth=1.5,
        label='Holt-Winters — RMSE 18.5K t', zorder=5)

# Prophet
ax.plot([x_hist[-1]] + x_fore,
        [hist_prod[-1]] + prophet_fore,
        color=C_PROPH, linewidth=2.5, linestyle='-',
        marker='o', markersize=6,
        markerfacecolor=C_PROPH,
        markeredgecolor='white', markeredgewidth=1.5,
        label='Prophet — RMSE 10.7K t (best fit)', zorder=5)

# Peak reference line
peak_val = max(hist_prod)
ax.axhline(y=peak_val, color=C_PEAK, linewidth=1.5,
           linestyle=':', alpha=0.7,
           label=f'FY 23/24 peak ({peak_val:.0f}K t)', zorder=3)
ax.text(x_all[-1] + 0.08, peak_val + 1.5,
        f'Peak\n{peak_val:.0f}K',
        ha='left', fontsize=8,
        color=C_PEAK, fontweight='bold')

# Divider
ax.axvline(x=mid, color=C_MUTED,
           linewidth=1.2, linestyle='--', alpha=0.4)

# Consensus annotation
ax.text(0.38, 0.08,
        'All three models confirm: no return to FY 23/24 peak through 2028',
        transform=ax.transAxes, fontsize=9,
        color='#8A5B0A', fontweight='bold', style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF0D7',
                  edgecolor='#8A5B0A', alpha=0.9))

# Axes
ax.set_xticks(x_all)
ax.set_xticklabels(all_labels, fontsize=9.5, color=C_MUTED)
ax.set_xlim(-0.5, len(x_all) - 0.3)

ymin = min(hist_prod + arima_fore + hw_fore +
           prophet_fore + prophet_lower) - 15
ymax = max(hist_prod + prophet_upper) + 25
ax.set_ylim(ymin, ymax)

ax.set_ylabel('SA Horticulture Production (K tonnes)',
              fontsize=10, color=C_MUTED, labelpad=10)
ax.set_xlabel('Financial Year',
              fontsize=10, color=C_MUTED, labelpad=8)
ax.tick_params(colors=C_MUTED, labelsize=9.5, length=0)
ax.yaxis.grid(True, color=C_GRID, linewidth=0.8)
ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color('#C0C0C0')

# Title
ax.set_title(
    'SA Horticulture Production Forecast \u2014 '
    'ARIMA(1,1,0), Holt-Winters and Prophet (FY 24/25\u201329/30)',
    fontsize=12, fontweight='bold', color=C_TITLE, pad=14)
ax.text(0.0, 1.02,
        'No model projects return to FY 23/24 peak  \u00b7  '
        'Supply constraint persistent through 2028  \u00b7  '
        'Source: ABS 2025; BOM 2026',
        transform=ax.transAxes,
        fontsize=8.5, color=C_MUTED, style='italic')

ax.legend(fontsize=9, loc='upper left',
          framealpha=0.92, edgecolor='#E2E8F0', fancybox=False)

plt.tight_layout()
plt.savefig('3model_forecast.png', dpi=180,
            bbox_inches='tight', facecolor=C_BG)

try:
    from IPython.display import Image, display
    display(Image('3model_forecast.png'))
except:
    pass

plt.show()
print("Saved: 3model_forecast.png")
